# One-Click Earthquake Risk Prediction
Bu notebook tek hucre akisi ile lokasyon + tarih araliginda riskli dakikalari listeler.

In [ ]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'mine' else Path.cwd()
PIPE = ROOT / 'pipeline'
if str(PIPE) not in sys.path:
    sys.path.insert(0, str(PIPE))

from config import DATA_RAW, MODELS
from generate_planet_features import build_feature_row, load_kernels
from seismic_features import compute_history_features
from train_location_models import location_features
from coord_dms import parse_coordinate, format_lat_dms, format_lon_dms

lat_dms = '39.55.48N'
lon_dms = '32.51.00E'
lat = parse_coordinate(lat_dms, is_lat=True)
lon = parse_coordinate(lon_dms, is_lat=False)
start = pd.Timestamp('2026-04-01T00:00:00Z')
end = pd.Timestamp('2026-04-03T00:00:00Z')
top_k = 20

minutes = pd.date_range(start=start, end=end, freq='min', tz='UTC')

clf = joblib.load(MODELS / 'location_classifier.joblib')
calibrator = joblib.load(MODELS / 'location_classifier_calibrator.joblib')
ensemble = joblib.load(MODELS / 'location_classifier_ensemble.joblib')
reg = joblib.load(MODELS / 'magnitude_regressor.joblib')
reg_q10 = joblib.load(MODELS / 'magnitude_regressor_q10.joblib')
reg_q90 = joblib.load(MODELS / 'magnitude_regressor_q90.joblib')
feature_cols = json.loads((MODELS / 'location_models_meta.json').read_text())['feature_columns']

rows = []
loc = location_features(lat, lon)
load_kernels()
try:
    for ts in minutes:
        r = build_feature_row(ts)
        r.update(loc)
        rows.append(r)
finally:
    import spiceypy as spice
    spice.kclear()

df = pd.DataFrame(rows)
df['time_utc'] = pd.to_datetime(df['time_utc'], utc=True)
eq = pd.read_parquet(DATA_RAW / 'earthquakes_m4_2000_2026.parquet')
eq['time_utc'] = pd.to_datetime(eq['time_utc'], utc=True).dt.floor('min')
hist = compute_history_features(df[['time_utc']].assign(latitude=lat, longitude=lon), eq)
df = pd.concat([df.reset_index(drop=True), hist.reset_index(drop=True)], axis=1)

X = df[feature_cols]
p_cal = calibrator.predict_proba(X)[:, 1]
ens_probs = np.vstack([m.predict_proba(X)[:, 1] for m in ensemble])
p_std = ens_probs.std(axis=0)

df['p_eq_m4_plus'] = p_cal
df['p_eq_m4_plus_low95'] = np.clip(p_cal - 1.96 * p_std, 0, 1)
df['p_eq_m4_plus_high95'] = np.clip(p_cal + 1.96 * p_std, 0, 1)
df['pred_magnitude_if_event'] = reg.predict(X)
df['pred_magnitude_q10'] = reg_q10.predict(X)
df['pred_magnitude_q90'] = reg_q90.predict(X)
df['latitude_dms'] = format_lat_dms(lat)
df['longitude_dms'] = format_lon_dms(lon)

result = df.sort_values('p_eq_m4_plus', ascending=False).head(top_k)
out = ROOT / 'models' / 'one_click_predictions.csv'
result.to_csv(out, index=False)
print(f'saved -> {out}')
result[['time_utc','latitude_dms','longitude_dms','p_eq_m4_plus','p_eq_m4_plus_low95','p_eq_m4_plus_high95','pred_magnitude_if_event','pred_magnitude_q10','pred_magnitude_q90']]